# Import Libraries

In [ ]:
import os
import ast
import json
import time
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score
from transformers import RobertaTokenizerFast, RobertaModel, get_linear_schedule_with_warmup

# CONFIG - edit these paths / hyperparameters, then just run the file

In [ ]:
from google.colab import files
uploaded = files.upload()

TRAIN_PATH = "/content/Training_clean.csv"
TEST_PATH  = "/content/Testing_clean.csv"
VAL_PATH   = "/content/Validation_clean.csv"

OUT_DIR = "/content/checkpoints"
PRED_OUT_PATH = "/content/predictions_test.csv"
METRICS_OUT_PATH = "/content/final_metrics_report.csv"

EPOCHS = 2
BATCH_SIZE = 4
LR = 2e-5
MAX_LEN = 128
MAX_TURNS = 5
CTX_LOSS_WEIGHT = 0.5
SEED = 42
MODEL_NAME = "roberta-base"

# FIX: SUBSET was used later in MAIN (`if SUBSET:`) but was never defined,
# which would have caused a second NameError right after the collate_fn one.
# Set to None to use the full dataset, or an int (e.g. 200) to quickly debug
# with a small slice of train/val/test.
SUBSET = None

Saving Testing_clean.csv to Testing_clean.csv
Saving Training_clean.csv to Training_clean.csv
Saving Validation_clean.csv to Validation_clean.csv


# CSV Load + Basic Check

In [ ]:
train_df = pd.read_csv("/content/Training_clean.csv")
test_df  = pd.read_csv("/content/Testing_clean.csv")
val_df   = pd.read_csv("/content/Validation_clean.csv")

print("Train rows:", len(train_df))
print("Test rows :", len(test_df))
print("Val rows  :", len(val_df))

Train rows: 11118
Test rows : 1000
Val rows  : 1000


# LABEL SCHEME (fixed by the dataset - do not remap to strings)

In [ ]:
NUM_ACTS = 5          # raw act ids are 1..4; index 0 unused but kept so
                       # raw_id == class_index (no string remapping anywhere)
NUM_EMOTIONS = 7        # raw emotion ids are 0..6

CONTEXT_STATES = [
    "achievement", "ongoing_problem", "emotional_support", "relationship_conflict",
    "health_concern", "academic_pressure", "loneliness", "financial_stress",
]
CTX2ID = {c: i for i, c in enumerate(CONTEXT_STATES)}
ID2CTX = {i: c for c, i in CTX2ID.items()}

# Data

In [ ]:
def load_conversations(df):
    """Turn a loaded DataFrame (id/acts/emotions/utterances/context_state
    columns) into a list of dicts, one per conversation. acts/emotions are
    kept as lists of ints exactly as given (ground truth) - never
    remapped to strings."""
    rows = []
    for _, r in df.iterrows():
        rows.append({
            "id": r["id"],
            "utterances": ast.literal_eval(r["utterances"]),
            "acts": [int(x) for x in ast.literal_eval(r["acts"])],
            "emotions": [int(x) for x in ast.literal_eval(r["emotions"])],
            "context_state": r["context_state"],
        })
    return rows


class DialogueDataset(Dataset):

    def __init__(self, conversations, tokenizer, max_len=MAX_LEN, max_turns=MAX_TURNS):
        self.conversations = conversations
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.max_turns = max_turns

    def __len__(self):
        return len(self.conversations)

    def __getitem__(self, idx):
        conv = self.conversations[idx]
        utts = conv["utterances"][: self.max_turns]
        acts = conv["acts"][: self.max_turns]
        emos = conv["emotions"][: self.max_turns]

        n = len(utts)

        enc = self.tokenizer(
            utts,
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt",
        )

        pad_turns = self.max_turns - n

        if pad_turns > 0:
            pad_ids = torch.full(
                (pad_turns, self.max_len),
                self.tokenizer.pad_token_id,
                dtype=torch.long
            )
            pad_mask = torch.zeros((pad_turns, self.max_len), dtype=torch.long)

            input_ids = torch.cat([enc["input_ids"], pad_ids], dim=0)
            attn_mask = torch.cat([enc["attention_mask"], pad_mask], dim=0)
        else:
            input_ids = enc["input_ids"]
            attn_mask = enc["attention_mask"]

        act_labels = torch.full((self.max_turns,), -100, dtype=torch.long)
        emo_labels = torch.full((self.max_turns,), -100, dtype=torch.long)

        act_labels[:n] = torch.tensor(acts, dtype=torch.long)
        emo_labels[:n] = torch.tensor(emos, dtype=torch.long)

        turn_mask = torch.zeros((self.max_turns,), dtype=torch.bool)
        turn_mask[:n] = True

        ctx_label = torch.tensor(CTX2ID[conv["context_state"]], dtype=torch.long)

        return {
            "input_ids": input_ids,
            "attention_mask": attn_mask,
            "act_labels": act_labels,
            "emo_labels": emo_labels,
            "ctx_label": ctx_label,
            # FIX: was "turn _mask" (stray space) -> Python syntax/key error.
            "turn_mask": turn_mask,
            "n_turns": n,
            "id": conv["id"],
        }


# FIX: collate_fn was used in every DataLoader(...) call but was never
# defined anywhere, which is exactly what caused your NameError.
# It just stacks the per-sample dict fields from DialogueDataset.__getitem__
# into batched tensors (and keeps n_turns / id as plain python lists, since
# run_epoch / predict_and_save index/iterate them per-sample).
def collate_fn(batch):
    return {
        "input_ids": torch.stack([b["input_ids"] for b in batch]),
        "attention_mask": torch.stack([b["attention_mask"] for b in batch]),
        "act_labels": torch.stack([b["act_labels"] for b in batch]),
        "emo_labels": torch.stack([b["emo_labels"] for b in batch]),
        "ctx_label": torch.stack([b["ctx_label"] for b in batch]),
        "turn_mask": torch.stack([b["turn_mask"] for b in batch]),
        "n_turns": [b["n_turns"] for b in batch],
        "id": [b["id"] for b in batch],
    }


# MODEL

In [ ]:
class MultiTaskRoberta(nn.Module):

    def __init__(self, model_name=MODEL_NAME, dropout=0.1):
        super().__init__()

        self.encoder = RobertaModel.from_pretrained(model_name)
        hidden = self.encoder.config.hidden_size

        self.dropout = nn.Dropout(dropout)

        self.act_head = nn.Linear(hidden, NUM_ACTS)
        self.emo_head = nn.Linear(hidden, NUM_EMOTIONS)
        self.ctx_head = nn.Linear(hidden, len(CONTEXT_STATES))

    def forward(self, input_ids, attention_mask, turn_mask):
        # input_ids, attention_mask: (B, T, L)
        B, T, L = input_ids.shape
        flat_ids = input_ids.view(B * T, L)
        flat_mask = attention_mask.view(B * T, L)

        out = self.encoder(input_ids=flat_ids, attention_mask=flat_mask)
        cls_emb = out.last_hidden_state[:, 0, :]      # (B*T, H)
        cls_emb = self.dropout(cls_emb).view(B, T, -1)  # (B, T, H)

        act_logits = self.act_head(cls_emb)             # (B, T, NUM_ACTS)
        emo_logits = self.emo_head(cls_emb)              # (B, T, NUM_EMOTIONS)

        mask_f = turn_mask.unsqueeze(-1).float()          # (B, T, 1)
        summed = (cls_emb * mask_f).sum(dim=1)              # (B, H)
        counts = mask_f.sum(dim=1).clamp(min=1.0)
        conv_emb = summed / counts
        ctx_logits = self.ctx_head(conv_emb)                # (B, NUM_CTX)

        return act_logits, emo_logits, ctx_logits


# TRAIN / EVAL LOOP

In [ ]:
def run_epoch(model, loader, optimizer, scheduler, device, ctx_loss_weight, train=True):
    model.train() if train else model.eval()
    act_loss_fn = nn.CrossEntropyLoss(ignore_index=-100)
    emo_loss_fn = nn.CrossEntropyLoss(ignore_index=-100)
    ctx_loss_fn = nn.CrossEntropyLoss()

    total_loss = 0.0
    all_act_preds, all_act_true = [], []
    all_emo_preds, all_emo_true = [], []
    all_ctx_preds, all_ctx_true = [], []

    context = torch.enable_grad() if train else torch.no_grad()
    with context:
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            act_labels = batch["act_labels"].to(device)
            emo_labels = batch["emo_labels"].to(device)
            ctx_label = batch["ctx_label"].to(device)
            turn_mask = batch["turn_mask"].to(device)

            if train:
                optimizer.zero_grad()

            act_logits, emo_logits, ctx_logits = model(input_ids, attention_mask, turn_mask)

            B, T, _ = act_logits.shape
            act_loss = act_loss_fn(act_logits.view(B * T, -1), act_labels.view(B * T))
            emo_loss = emo_loss_fn(emo_logits.view(B * T, -1), emo_labels.view(B * T))
            ctx_loss = ctx_loss_fn(ctx_logits, ctx_label)
            loss = act_loss + emo_loss + ctx_loss_weight * ctx_loss

            if train:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()

            total_loss += loss.item()

            valid = act_labels.view(-1) != -100
            all_act_preds.extend(act_logits.view(B * T, -1).argmax(-1)[valid].cpu().tolist())
            all_act_true.extend(act_labels.view(-1)[valid].cpu().tolist())
            all_emo_preds.extend(emo_logits.view(B * T, -1).argmax(-1)[valid].cpu().tolist())
            all_emo_true.extend(emo_labels.view(-1)[valid].cpu().tolist())
            all_ctx_preds.extend(ctx_logits.argmax(-1).cpu().tolist())
            all_ctx_true.extend(ctx_label.cpu().tolist())

    return {
        "loss": total_loss / max(len(loader), 1),
        "act_acc": accuracy_score(all_act_true, all_act_preds),
        "act_f1": f1_score(all_act_true, all_act_preds, average="macro", zero_division=0),
        "emo_acc": accuracy_score(all_emo_true, all_emo_preds),
        "emo_f1": f1_score(all_emo_true, all_emo_preds, average="macro", zero_division=0),
        "ctx_acc": accuracy_score(all_ctx_true, all_ctx_preds),
        "ctx_f1": f1_score(all_ctx_true, all_ctx_preds, average="macro", zero_division=0),
    }


def evaluate_split(model, conversations, tokenizer, device, batch_size=BATCH_SIZE,
                    max_len=MAX_LEN, max_turns=MAX_TURNS, ctx_loss_weight=CTX_LOSS_WEIGHT):
    """Compute act/emotion/context_state accuracy + macro-F1 for one split
    (a list of conversations already produced by load_conversations)."""
    ds = DialogueDataset(conversations, tokenizer, max_len, max_turns)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)
    metrics = run_epoch(model, loader, optimizer=None, scheduler=None, device=device,
                         ctx_loss_weight=ctx_loss_weight, train=False)
    metrics["n_conversations"] = len(conversations)
    return metrics


def print_metrics_table(name_to_metrics):
    """Pretty-print a side-by-side accuracy table for train/val/test."""
    cols = list(name_to_metrics.keys())
    rows = [
        ("conversations", "n_conversations", "{:.0f}"),
        ("act accuracy", "act_acc", "{:.4f}"),
        ("act macro-F1", "act_f1", "{:.4f}"),
        ("emotion accuracy", "emo_acc", "{:.4f}"),
        ("emotion macro-F1", "emo_f1", "{:.4f}"),
        ("context accuracy", "ctx_acc", "{:.4f}"),
        ("context macro-F1", "ctx_f1", "{:.4f}"),
    ]
    header = f"{'metric':<20}" + "".join(f"{c:>14}" for c in cols)
    print(header)
    print("-" * len(header))
    for label, key, fmt in rows:
        line = f"{label:<20}"
        for c in cols:
            val = name_to_metrics[c].get(key, float("nan"))
            line += f"{fmt.format(val):>14}"
        print(line)


# PREDICT (writes back conversation-level rows with int pred_acts / pred_emotions)

In [ ]:
def predict_and_save(model, df_raw, conversations, tokenizer, device, out_path,
                      batch_size=BATCH_SIZE, max_len=MAX_LEN, max_turns=MAX_TURNS):
    ds = DialogueDataset(conversations, tokenizer, max_len, max_turns)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

    model.eval()
    id_to_pred = {}
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            turn_mask = batch["turn_mask"].to(device)

            act_logits, emo_logits, ctx_logits = model(input_ids, attention_mask, turn_mask)
            act_preds = act_logits.argmax(-1).cpu()
            emo_preds = emo_logits.argmax(-1).cpu()
            ctx_preds = ctx_logits.argmax(-1).cpu()

            for i, conv_id in enumerate(batch["id"]):
                n = batch["n_turns"][i]
                id_to_pred[conv_id] = {
                    "pred_acts": [int(x) for x in act_preds[i][:n].tolist()],
                    "pred_emotions": [int(x) for x in emo_preds[i][:n].tolist()],
                    "pred_context_state": ID2CTX[int(ctx_preds[i].item())],
                }

    df_out = df_raw.copy()
    df_out["pred_acts"] = df_out["id"].map(lambda i: json.dumps(id_to_pred[i]["pred_acts"]))
    df_out["pred_emotions"] = df_out["id"].map(lambda i: json.dumps(id_to_pred[i]["pred_emotions"]))
    df_out["pred_context_state"] = df_out["id"].map(lambda i: id_to_pred[i]["pred_context_state"])

    df_out.to_csv(out_path, index=False)
    print(f"Wrote {len(df_out)} rows to {out_path}")
    print(df_out[["id", "acts", "pred_acts", "emotions", "pred_emotions",
                   "context_state", "pred_context_state"]].head(3).to_string())
    return df_out

# Sanity check: Verify whether the full values are loaded correctly or if any truncation has occurred.


In [ ]:
# Reload CSVs directly
train_raw = pd.read_csv(TRAIN_PATH)
val_raw   = pd.read_csv(VAL_PATH)
test_raw  = pd.read_csv(TEST_PATH)

# Disable pandas truncation
pd.set_option("display.max_colwidth", None)

for name, df in [("TRAIN", train_raw), ("VAL", val_raw), ("TEST", test_raw)]:
    row = df.iloc[0]

    utt_list = ast.literal_eval(str(row["utterances"]))
    act_list = ast.literal_eval(str(row["acts"]))
    emo_list = ast.literal_eval(str(row["emotions"]))

    print("\n" + "=" * 80)
    print(f"{name} | Row ID: {row['id']}")
    print("=" * 80)

    print("Utterances Count:", len(utt_list))
    print("Acts Count      :", len(act_list))
    print("Emotions Count  :", len(emo_list))
    print("Context State   :", row["context_state"])

    print("\nFULL Utterances:")
    print(utt_list)

    print("\nFULL Acts:")
    print(act_list)

    print("\nFULL Emotions:")
    print(emo_list)

    assert len(utt_list) == len(act_list) == len(emo_list), \
        f"Length mismatch found in {name}!"

print("\n✅ All datasets checked successfully.")
print("✅ No truncation detected.")
print("✅ utterances, acts, emotions lengths match.")


TRAIN | Row ID: a438b751ab9997cdb35f07bfe3dfb010b96365f4762d77f87e5f41290ff61c3d_0
Utterances Count: 10
Acts Count      : 10
Emotions Count  : 10
Context State   : achievement

FULL Utterances:
['Say , Jim , how about going for a few beers after dinner ?', 'You know that is tempting but is really not good for our fitness .', 'What do you mean ? It will help us to relax .', "Do you really think so ? I don't . It will just make us fat and act silly . Remember last time ?", "I guess you are right.But what shall we do ? I don't feel like sitting at home .", 'I suggest a walk over to the gym where we can play singsong and meet some of our friends .', "That's a good idea . I hear Mary and Sally often go there to play pingpong.Perhaps we can make a foursome with them .", 'Sounds great to me ! If they are willing , we could ask them to go dancing with us.That is excellent exercise and fun , too .', "Good.Let ' s go now .", 'All right .']

FULL Acts:
[3, 4, 2, 2, 2, 3, 4, 1, 3, 4]

FULL Emotio

# MAIN - runs top to bottom: load -> train -> evaluate (train/val/test) -> predict

In [ ]:
if __name__ == "__main__":
    import random

    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    # ---- Load data ----
    train_raw = pd.read_csv(TRAIN_PATH, usecols=["id", "acts", "emotions", "utterances", "context_state"])
    test_raw  = pd.read_csv(TEST_PATH,  usecols=["id", "acts", "emotions", "utterances", "context_state"])
    val_raw   = pd.read_csv(VAL_PATH,   usecols=["id", "acts", "emotions", "utterances", "context_state"])

    print("Loaded -> Train:", len(train_raw))
    print("Loaded -> Test :", len(test_raw))
    print("Loaded -> Val  :", len(val_raw))

    train_convs = load_conversations(train_raw)
    val_convs   = load_conversations(val_raw)
    test_convs  = load_conversations(test_raw)

    if SUBSET:
        train_convs = train_convs[:SUBSET]
        val_convs   = val_convs[: max(2, SUBSET // 5)]
        test_convs  = test_convs[: max(2, SUBSET // 5)]
        print(f"(SUBSET={SUBSET} active)")

    os.makedirs(OUT_DIR, exist_ok=True)

    # ---- Tokenizer + Data ----
    tokenizer = RobertaTokenizerFast.from_pretrained(MODEL_NAME)

    train_ds = DialogueDataset(train_convs, tokenizer, MAX_LEN, MAX_TURNS)
    val_ds   = DialogueDataset(val_convs,   tokenizer, MAX_LEN, MAX_TURNS)
    test_ds  = DialogueDataset(test_convs,  tokenizer, MAX_LEN, MAX_TURNS)  # ← added

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  collate_fn=collate_fn)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
    test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)  # ← added

    # ---- Model ----
    model = MultiTaskRoberta(MODEL_NAME).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

    total_steps = len(train_loader) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(0.06 * total_steps),
        num_training_steps=total_steps
    )

    best_val_score = -1
    best_ckpt_path = os.path.join(OUT_DIR, "best_model.pt")

    # ---- Train ----
    for epoch in range(1, EPOCHS + 1):
        t0 = time.time()

        tr = run_epoch(model, train_loader, optimizer, scheduler, device, CTX_LOSS_WEIGHT, train=True)
        va = run_epoch(model, val_loader,   optimizer, scheduler, device, CTX_LOSS_WEIGHT, train=False)
        te = run_epoch(model, test_loader,  optimizer, scheduler, device, CTX_LOSS_WEIGHT, train=False)  # ← added

        dt = time.time() - t0

        print(f"\nEpoch {epoch}/{EPOCHS} ({dt:.1f}s)")
        print(f"  train: loss={tr['loss']:.4f}  act_acc={tr['act_acc']:.4f}  act_f1={tr['act_f1']:.4f}  emo_acc={tr['emo_acc']:.4f}  emo_f1={tr['emo_f1']:.4f}  ctx_acc={tr['ctx_acc']:.4f}  ctx_f1={tr['ctx_f1']:.4f}")
        print(f"  val:   loss={va['loss']:.4f}  act_acc={va['act_acc']:.4f}  act_f1={va['act_f1']:.4f}  emo_acc={va['emo_acc']:.4f}  emo_f1={va['emo_f1']:.4f}  ctx_acc={va['ctx_acc']:.4f}  ctx_f1={va['ctx_f1']:.4f}")
        print(f"  test:  loss={te['loss']:.4f}  act_acc={te['act_acc']:.4f}  act_f1={te['act_f1']:.4f}  emo_acc={te['emo_acc']:.4f}  emo_f1={te['emo_f1']:.4f}  ctx_acc={te['ctx_acc']:.4f}  ctx_f1={te['ctx_f1']:.4f}")

        val_score = va["act_f1"] + va["emo_f1"] + va["ctx_f1"]

        if val_score > best_val_score:
            best_val_score = val_score
            torch.save(
                {"model_state_dict": model.state_dict(), "epoch": epoch, "val_metrics": va},
                best_ckpt_path
            )
            print("  -> saved best model")

    print("\nBest score:", best_val_score)

    # ---- Load best ----
    print("\nLoading best model...")
    best_ckpt = torch.load(best_ckpt_path, map_location=device)
    model.load_state_dict(best_ckpt["model_state_dict"])
    model.eval()

    # ---- Evaluate ----
    print("Evaluating train/val/test...")

    train_metrics = evaluate_split(model, train_convs, tokenizer, device)
    val_metrics   = evaluate_split(model, val_convs,   tokenizer, device)
    test_metrics  = evaluate_split(model, test_convs,  tokenizer, device)

    results = {"train": train_metrics, "validation": val_metrics, "test": test_metrics}

    print_metrics_table(results)

    pd.DataFrame(results).to_csv(METRICS_OUT_PATH)
    print(f"Saved metrics to {METRICS_OUT_PATH}")

    # ---- Predict ----
    print("Running predictions...")
    predict_and_save(model, test_raw, test_convs, tokenizer, device, PRED_OUT_PATH)

Using device: cuda
Loaded -> Train: 11118
Loaded -> Test : 1000
Loaded -> Val  : 1000


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Epoch 1/2 (1312.7s)
  train: loss=1.8128  act_acc=0.7454  act_f1=0.4338  emo_acc=0.8403  emo_f1=0.1800  ctx_acc=0.5244  ctx_f1=0.1446
  val:   loss=1.5552  act_acc=0.7528  act_f1=0.6478  emo_acc=0.8592  emo_f1=0.2225  ctx_acc=0.6190  ctx_f1=0.1613
  test:  loss=1.6593  act_acc=0.7997  act_f1=0.6673  emo_acc=0.7997  emo_f1=0.2360  ctx_acc=0.5730  ctx_f1=0.1520
  -> saved best model

Epoch 2/2 (1310.6s)
  train: loss=1.5184  act_acc=0.8007  act_f1=0.6549  emo_acc=0.8611  emo_f1=0.2400  ctx_acc=0.5782  ctx_f1=0.1531
  val:   loss=1.3688  act_acc=0.7691  act_f1=0.6710  emo_acc=0.9056  emo_f1=0.2769  ctx_acc=0.6640  ctx_f1=0.1783
  test:  loss=1.4922  act_acc=0.8131  act_f1=0.6915  emo_acc=0.8475  emo_f1=0.2576  ctx_acc=0.6020  ctx_f1=0.1600
  -> saved best model

Best score: 1.1261734567216608

Loading best model...
Evaluating train/val/test...
metric                       train    validation          test
--------------------------------------------------------------
conversations       

# FINAL SUMMARY

In [ ]:
summary = pd.DataFrame({
    "Dataset": ["Train", "Validation", "Test"],

    "Act_Accuracy (%)": [
        train_metrics["act_acc"] * 100,
        val_metrics["act_acc"] * 100,
        test_metrics["act_acc"] * 100,
    ],

    "Emotion_Accuracy (%)": [
        train_metrics["emo_acc"] * 100,
        val_metrics["emo_acc"] * 100,
        test_metrics["emo_acc"] * 100,
    ],

    "Context_Accuracy (%)": [
        train_metrics["ctx_acc"] * 100,
        val_metrics["ctx_acc"] * 100,
        test_metrics["ctx_acc"] * 100,
    ],

    "Act_F1 (%)": [
        train_metrics["act_f1"] * 100,
        val_metrics["act_f1"] * 100,
        test_metrics["act_f1"] * 100,
    ],

    "Emotion_F1 (%)": [
        train_metrics["emo_f1"] * 100,
        val_metrics["emo_f1"] * 100,
        test_metrics["emo_f1"] * 100,
    ],

    "Context_F1 (%)": [
        train_metrics["ctx_f1"] * 100,
        val_metrics["ctx_f1"] * 100,
        test_metrics["ctx_f1"] * 100,
    ]
})

summary = summary.round(2)

print("\n===== FINAL RESULTS (%) =====")
print(summary)


===== FINAL RESULTS (%) =====
      Dataset  Act_Accuracy (%)  Emotion_Accuracy (%)  Context_Accuracy (%)  \
0       Train             81.77                 86.53                 60.93   
1  Validation             76.91                 90.56                 66.40   
2        Test             81.31                 84.75                 60.20   

   Act_F1 (%)  Emotion_F1 (%)  Context_F1 (%)  
0       70.08           28.67           16.57  
1       67.10           27.69           17.83  
2       69.15           25.76           16.00  
